# PatchCore anomalib backend — Colab driver

Phases 0–2 of `docs/superpowers/plans/2026-07-24-patchcore-anomalib-backend-colab.md`.
**The playbook is the authority**; this notebook is its runnable form. Phases 3 (the VisA ±1pt
gate) and 4 (freeze provenance) stay in the playbook because they need a loader and a published
table that cannot be pre-written — see the last section.

**Runtime → Change runtime type → T4 GPU** before running anything.

## What this session must prove, in order

| Phase | Proves | Gate |
|---|---|---|
| 0 | anomalib version + API shape | if 0.3 mismatches, **stop and adapt Phase 1** |
| 1 | the backend works | the smoke test, not the docs |
| 2 | it drives the whole runner path | I-AUROC well above 0.5 on `regular` |

**Honest framing, from the playbook:** every anomalib call here comes from the library's published
API, but the exact `>=1.1` signatures were never executed in the environment that wrote it. Step 0.3
and the Phase 1 smoke test are the real verification. **A mismatch there is expected maintenance,
not a plan failure** — adapt the call to the installed version and continue. The one thing that is
never adapted away is the ±1.0 VisA gate.

## Phase 0 — environment and API verification

### 0.1 GPU + repo

In [ ]:
!nvidia-smi -L      # must print a GPU; if not: Runtime -> Change runtime type -> T4 GPU

REPO = "/content/vlm-anomaly-bench"
!git clone https://github.com/andrudebaran7/vlm-anomaly-bench.git {REPO}
%cd {REPO}

# NOT --quiet: a failed editable install here surfaces four cells later as a baffling
# "No module named 'vlmab'", and the quiet flag is what hides the real reason.
!pip install -e .

# Belt and braces: the repo uses a src/ layout, so if a later dependency install disturbs
# the editable-install finder, this keeps vlmab importable anyway.
import sys
if REPO + "/src" not in sys.path:
    sys.path.insert(0, REPO + "/src")

import vlmab
print("vlmab imports OK from", vlmab.__file__)

### 0.2 Install anomalib and record the resolved version

The version this prints is provenance: it goes into `configs/methods/patchcore_ref.yaml` in Phase 4,
recorded as the version the VisA gate passed on.

In [ ]:
!pip install "anomalib>=1.1"

import anomalib, torch
ANOMALIB_VERSION = anomalib.__version__
print("anomalib", ANOMALIB_VERSION, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU - the backend cannot run"

# Provenance: this exact string goes into configs/methods/patchcore_ref.yaml in Phase 4,
# recorded as the version the VisA gate passed on. If the runtime restarts, this variable
# is lost with it - re-run this cell before Phase 4 rather than reconstructing it.

### ⚠️ If Colab offered to restart the runtime — read this

Installing anomalib pulls a large dependency tree that often conflicts with Colab's preinstalled
numpy/torch, so Colab commonly prompts **"Restart runtime"**. Accepting is usually the right call,
but a restart throws away:

- the `%cd` from 0.1, so the working directory reverts to `/content`;
- every Python variable, including `ANOMALIB_VERSION`;
- sometimes the editable install's path hook.

Files on disk survive — the cloned repo and anything `%%writefile` already wrote are still there.

**Run the cell below after any restart.** It is idempotent, so running it when you did not restart
costs nothing.

In [ ]:
# Idempotent restart recovery: safe to run at any point, any number of times.
%cd /content/vlm-anomaly-bench

import sys
if "/content/vlm-anomaly-bench/src" not in sys.path:
    sys.path.insert(0, "/content/vlm-anomaly-bench/src")

import vlmab, anomalib, torch
ANOMALIB_VERSION = anomalib.__version__          # restore the provenance string
print("vlmab OK | anomalib", ANOMALIB_VERSION,
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

import os
print("backend file written:",
      os.path.isfile("src/vlmab/methods/patchcore_backend.py"),
      "(False before Phase 1.1 - that is expected)")

### 0.3 VERIFY the API against the installed version

**This is the cheapest place to catch API drift, and the single most likely failure of the plan.**
Read each output and confirm it matches what Phase 1 assumes. Note any mismatch — you adapt Phase 1
to the installed signature, never the other way round.

In [ ]:
import inspect
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.data import Folder

# (a) constructor takes the pre-registered hyperparameters?
#     expect: backbone, layers, coreset_sampling_ratio, num_neighbors
print("(a) Patchcore.__init__:\n", inspect.signature(Patchcore.__init__), "\n")

# (b) the default pre-processor transform score() will use?
#     expect: Resize(256) -> CenterCrop(224) -> Normalize(imagenet)
print("(b) pre-processor transform:\n", Patchcore.configure_pre_processor().transform, "\n")

# (c) Folder accepts normal-only with splits disabled?
#     expect: name, root, normal_dir, val_split_mode, test_split_mode, task
print("(c) Folder.__init__:\n", inspect.signature(Folder.__init__), "\n")

# (d) Engine kwargs the backend passes?
#     expect: accelerator, devices, max_epochs, logger, enable_checkpointing
print("(d) Engine.__init__:\n", inspect.signature(Engine.__init__))

**Record your findings before moving on.** (d) is a convenience setting — a rejected kwarg gets
dropped. (a) and (b) are correctness: a changed hyperparameter name or a different transform
changes the numbers.

| Check | Matched? | If not, what differs |
|---|---|---|
| (a) constructor | | |
| (b) transform | | |
| (c) Folder | | |
| (d) Engine | | |

### 1.1 Sync the backend from the repo

`src/vlmab/methods/patchcore_backend.py` is a **tracked file in the repo**, not something this
notebook writes. That matters for iteration: when a fix is pushed to `master`, you pick it up by
**re-running this cell** — no editing, no pasting, no reloading the notebook.

It also puts the source where source belongs. Code that lives only inside a notebook's JSON cannot
be diffed, reviewed, or imported by anything else.

In [ ]:
REPO = "/content/vlm-anomaly-bench"
!cd {REPO} && git pull

import sys, importlib
for p in (f"{REPO}/src", f"{REPO}/notebooks"):
    if p not in sys.path:
        sys.path.insert(0, p)

import vlmab.methods.patchcore_backend as _bk
importlib.reload(_bk)
PatchCoreBackend = _bk.PatchCoreBackend

import inspect
print(inspect.getsource(_bk.PatchCoreBackend.__init__))
print("--- backend synced from master; the source above is what will run ---")

### 1.1b Probe the backend stage by stage — run this when something breaks

`fit()` does five things behind one call, so a failure lands six frames deep in Lightning and says
nothing about which of *our* assumptions broke. `probe_patchcore` splits it into ten stages, each
declaring what it expects **before** running the smallest thing that tests it, stopping at the first
failure with the state that matters dumped.

Stage 1 is the one that earns its keep: it enumerates what `Folder`/`Engine`/`Patchcore` actually
accept and flags any `**kwargs` sink — a sink accepts an unknown argument at construction and defers
the failure to whoever consumes it, which is why `Engine(task=...)` built fine and then died inside
`fit()`.

The probe does **not** use `PatchCoreBackend`; it rebuilds each step independently. So it works
regardless of what state the backend file is in.

In [ ]:
import probe_patchcore, importlib
importlib.reload(probe_patchcore)

ctx = probe_patchcore.run()              # stops at the first failure
# ctx = probe_patchcore.run(keep_going=True)   # or: run every stage for the full picture

### 1.1c Stage 7 candidates — run this if the probe stops at `engine.train`

**What the probe found (2026-08-21, anomalib 2.6.0):**

```
AttributeError: 'Folder' object has no attribute 'val_data'
```

`val_split_mode=ValSplitMode.NONE` means the datamodule never builds `val_data`, but Lightning's
fit loop sets up the validation loop unconditionally before the first epoch. The two options are
incompatible, and no constructor argument fixes that.

Worth noticing what stage 7's own log says: *"configure_optimizers returned None, this fit will run
with no optimizer"*. PatchCore does not train — it runs a forward pass over the normal images and
subsamples a coreset. The Trainer is scaffolding around that.

`diagnose_train()` tests three candidates on a fresh model each, and reports which fill the memory
bank:

- **A** — keep every training image, tell Lightning not to validate (`limit_val_batches=0`).
  Preferred: PatchCore's memory bank *is* the model, so holding images back changes the result.
- **B** — give the datamodule a real validation split. Works, but costs training images.
- **C** — call the fit path only, skipping `Engine.train`'s test phase.

In [ ]:
import probe_patchcore, importlib
importlib.reload(probe_patchcore)

results = probe_patchcore.diagnose_train()

### 1.2 Smoke test — the real verification

Twenty flat-ish "normal" images and one with a bright injected patch. **The defect must score
higher.** If it does not, the fit or score path is wrong — debug here, on synthetic data, before
touching the real dataset.

The two calls the docs are least certain about are exercised for the first time here: `self._model(tensor)`
returning `.pred_score`/`.anomaly_map`, and the `Engine(...)` kwargs. If the model returns a
different container, adapt only the two extraction lines in `score()`.

> **`ModuleNotFoundError: No module named 'vlmab'` here** means the runtime restarted after the
> anomalib install (or the editable install failed). Run the restart-recovery cell in Phase 0, then
> re-run 1.1 if `patchcore_backend.py` is missing, then come back.

In [ ]:
import numpy as np
from vlmab.methods.patchcore_backend import PatchCoreBackend

rng = np.random.default_rng(0)
normal = [rng.integers(90, 110, (256, 256, 3), dtype=np.uint8) for _ in range(20)]  # flat-ish
anom = normal[0].copy(); anom[40:80, 40:80] = 255                                   # bright patch

b = PatchCoreBackend()
b.fit(iter(normal))
s_normal, m_normal = b.score(normal[1])
s_anom, m_anom = b.score(anom)

assert m_anom.ndim == 2 and np.isfinite(m_anom).all(), (m_anom.shape,)
assert isinstance(s_anom, float) and np.isfinite(s_anom)
assert s_anom > s_normal, (s_anom, s_normal)   # the defect must score higher
print("OK  normal:", round(s_normal, 4), " anomalous:", round(s_anom, 4), " map:", m_anom.shape)

### 1.3 Commit the backend

Set your git identity and a token first (fine-grained, `contents: write` on this repo). If you would
rather not paste a token into Colab, skip every commit cell and download the files at the end
instead.

In [ ]:
import os
from getpass import getpass

os.environ["GH_TOKEN"] = getpass("GitHub token (input hidden): ")
!git config user.email "sierprinsky@gmail.com"
!git config user.name "andrudebaran7"
!git remote set-url origin https://$GH_TOKEN@github.com/andrudebaran7/vlm-anomaly-bench.git
print("remote configured")

In [ ]:
!git add src/vlmab/methods/patchcore_backend.py
!git commit -m "feat: anomalib PatchCore backend bridging the fit/score seam (GPU-only)"
!git push

## Phase 2 — end to end through the existing runner, on Vial

Proves the backend drives the same path `intensity_baseline` already exercises —
`run_evaluation` → shard → `aggregate` — before spending a full grid.

### 2.1 Download and verify one category

Fetch Vial into `data/mvtec_ad2/vial` per `docs/datasets-access.md` (0.77 GB), then verify the
layout. **Never into `/tmp`** — it is tmpfs, i.e. RAM.

In [ ]:
# --- fetch Vial into data/mvtec_ad2/vial here (see docs/datasets-access.md) ---

!python scripts/prepare_data.py --root data/mvtec_ad2 --category vial
# Expected: exit 0, "OK", and the verified counts:
#   train regular=291, validation regular=41, test_public 140 across 7 conditions,
#   test_private 276, test_private_mixed 276

### 2.2 Run PatchCore over Vial's public test split

The runner calls `method.fit(train_images, "vial")` from the `train` split (protocol §3) before
scoring the 140 public-test images.

**Watch for the runner's float16 overflow guard.** If it fires, PatchCore's raw scores exceed 65504
— record the value and stop. That means the map needs a documented scale, which is a §4 protocol
amendment, not a silent change.

In [ ]:
from vlmab.datasets.mvtec_ad2 import MVTecAD2
from vlmab.eval.runner import run_evaluation
from vlmab.eval.store import ResultStore
from vlmab.eval.provenance import run_meta
from vlmab.methods.patchcore_ref import PatchCoreRef
from vlmab.methods.patchcore_backend import PatchCoreBackend

method = PatchCoreRef(backend=PatchCoreBackend())
store = ResultStore("results/patchcore/vial/shards")
meta = run_meta({"method": "patchcore_ref", "split": "test_public"}, seed=0)

run_evaluation(
    MVTecAD2("data/mvtec_ad2"), method, store, meta,
    categories=["vial"], split="test_public",
    maps_dir="results/patchcore/vial/maps",
)

### 2.3 Aggregate per lighting condition and sanity-check

Per lighting condition, not per category — that is what MVTec AD 2 exists to measure, and what keeps
native-resolution evaluation inside the memory budget.

**Expected:** I-AUROC **well above 0.5** on `regular`. A full-shot anchor that cannot beat chance on
its easiest condition is broken. A large drop from `regular` to the `shift_*` / `over/underexposed`
conditions is the lighting-robustness story, not an error — record the table either way.

In [ ]:
from vlmab.eval.store import ResultStore
from vlmab.eval.aggregate import aggregate

df = ResultStore("results/patchcore/vial/shards").load_all()
print(aggregate(df, by="meta_lighting")[
    ["meta_lighting", "i_auroc", "au_pro_030", "au_pro_005", "n"]].to_string(index=False))

## What comes next — not in this notebook, on purpose

### Phase 3 — the VisA ±1pt gate (the acceptance criterion)

**No MVTec AD 2 number is reported until PatchCore reproduces its published VisA image-AUROC within
±1.0 through this adapter and this repo's metrics** (protocol §2). Two pieces cannot be pre-written
and are yours to supply — the playbook marks both:

1. **A VisA loader.** Its layout is per-object `Data/Images/{Normal,Anomaly}` plus a split CSV —
   nothing like MVTec AD 2, so it does not use `MVTecAD2`.
2. **`PUBLISHED_VISA_IAUROC`**, the per-object table you measure against. **Record the exact source
   next to it**, and confirm it is image-AUROC, not pixel.

Then write `results/reproduction/patchcore_visa.md` with the table, the anomalib version, and the
pass/fail verdict.

If an object misses ±1.0, the likely causes in order: a preprocessing mismatch, the wrong published
baseline, or a coreset-ratio/backbone mismatch. If it genuinely cannot be reproduced, the anchor is
flagged as such in every table (protocol §2).

### Phase 4 — freeze provenance

Put the `ANOMALIB_VERSION` from 0.2 into `configs/methods/patchcore_ref.yaml`, commit this notebook,
push, and confirm both CI jobs (3.11, 3.13) stay green — CI never imports anomalib, so what it
exercises from this work is that `patchcore_backend.py` imports lazily.

### Then: the first real threshold calibration

Once the gate passes, PatchCore is the first *real* method that can produce a calibration artifact
(`intensity_baseline` is the floor, not a detector). That artifact is what lets the paper's C1/C3
`\todo{withdraw}` tripwires come out.

```bash
python scripts/run_eval.py --method patchcore_ref --root data/mvtec_ad2 \
    --results results/patchcore/val --maps-dir results/patchcore/val/maps \
    --split validation --category vial

python scripts/calibrate_threshold.py --results results/patchcore/val \
    --dataset mvtec_ad2 --method patchcore_ref \
    --out configs/thresholds/mvtec_ad2__patchcore_ref.yaml
```

Note the **separate results root**: shards are keyed `(dataset, method, category)` with no split in
the filename, so a validation run and a test_public run written to the same root collide. The CLI
refuses any split but `validation`, so a mistake here raises rather than silently calibrating on
test data.

Commit that YAML — committing it before any submission is what makes the pre-registration auditable.